# Introducción a OpenSeesPy - Viga Simplemente Apoyada
Este notebook modela una viga estáticamente determinada con una carga uniformemente distribuida $w$ utilizando elementos lineales elásticos (`elasticBeamColumn`).

**Objetivo:** Configurar las propiedades mecánicas elementales ($A$, $E$, $I$), aplicar una carga distribuida y extraer las reacciones y solicitaciones para validar el modelo.

## Importación de librerías

In [ ]:
import math
import sys

# 1. Verificación de entorno e instalación automática para Colab
if 'google.colab' in sys.modules:
    print("Entorno Google Colab detectado. Instalando OpenSeesPy y otras dependencias...")
    !pip install openseespy opsvis -q
    print("¡Instalación completada!")
else:
    print("Entorno local detectado. Usando librerías del sistema.")

# 2. Importación de módulos obligatorios
import openseespy.opensees as ops
print("OpenSeesPy importado exitosamente.")

## Definición de unidades

In [ ]:
# Longitudes
mm = 1
cm = 10*mm
m = 1000*mm

# Fuerza
N = 1
kN = 1000*N

# Presión
MPa = N / (mm ** 2)

## Propiedades de los materiales

In [ ]:
fc = 20*MPa
E = 4700 * math.sqrt(fc) * MPa        # Módulo de Elasticidad del hormigón(MPa)

## Propiedades geométricas

In [ ]:
# Definición de variables geométricas y materiales básicas
Lv = 6.00*m             # Longitud de la viga (m)

# Sección de hormigón
b = 20*cm               # Ancho de la viga de hormigón
d = 50*cm               # Altura de la viga de hormigón

A = b * d               # Área de la sección transversal
I = b * d**3 / 12       # Momento de Inercia respecto al eje Z - Requerido para flexión 2D

## Acciones sobre la estructura

In [ ]:
a_inf = 5.00*m                      # Ancho de influencia de las losas
D_losa = 3.00*kN/m**2 * a_inf       # Acciones de carga permanente de losas
D_viga = 25*kN/m**3 * A             # Acciones de peso propio de la viga
L = 2.00*kN/m**2 * a_inf            # Accionnes de sobrecarga

wu = 1.2 * (D_losa + D_viga) + 1.6 * L

## Modelo estructural

### Limpieza y definición de parámetros básicos

In [ ]:
# Limpiar el entorno de análisis previo
ops.wipe()

# Inicializar el modelo: 2 dimensiones (2D), 3 grados de libertad por nodo (Ux, Uy, Rz)
# ref: https://openseespydoc.readthedocs.io/en/latest/src/model.html#model
ops.model('basic', '-ndm', 2, '-ndf', 3)

# 1. Definición de Nodos (Coordenadas X, Y)
# ref: https://openseespydoc.readthedocs.io/en/latest/src/node.html#node
ops.node(1, 0.0, 0.0)  # Apoyo Izquierdo (X=0)
ops.node(2, Lv,   0.0)  # Apoyo Derecho   (X=L)

# 2. Condiciones de Contorno / Apoyos (1 = Restringido, 0 = Libre)
# ref: https://openseespydoc.readthedocs.io/en/latest/src/SP_Constraint.html#sp-constraint-commands
ops.fix(1, 1, 1, 0)    # Nodo 1: Fijo en X, Fijo en Y, Rotación Libre (Apoyo Fijo)
ops.fix(2, 0, 1, 0)    # Nodo 2: Libre en X, Fijo en Y, Rotación Libre (Apoyo Móvil)

# 3. Configuración Geométrica del Elemento (Transformación de coordenadas de local a global)
# En 2D lineal basta con asignarle una etiqueta (1) y el tipo 'Linear'
# ref: https://openseespydoc.readthedocs.io/en/latest/src/geomTransf.html#geomTransf
ops.geomTransf('Linear', 1)

# 4. Conectividad de Elementos (elasticBeamColumn)
# Argumentos: etiqueta_elemento, nodo_i, nodo_j, Área, E, Inercia, etiqueta_transformación
# ref: https://openseespydoc.readthedocs.io/en/latest/src/elasticBeamColumn.html#elastic-beam-column-element
ops.element('elasticBeamColumn', 1, 1, 2, A, E, I, 1)

print("Modelo estructural construido con éxito.")

### Aplicación de las acciones

In [ ]:
# 1. Crear el patrón de carga (Time Series lineal y Load Pattern estándar)
ops.timeSeries('Linear', 1) # ref: https://openseespydoc.readthedocs.io/en/latest/src/linearTs.html#linear-timeseries
ops.pattern('Plain', 1, 1) # ref: https://openseespydoc.readthedocs.io/en/latest/src/plainPattern.html#plain-pattern

# 2. Aplicar carga distribuida en el elemento 1
# Argumentos: 'eleLoad', '-ele', etiqueta_elemento, '-type', '-beamUniform', carga_w_dirección_y
# (Negativo porque la gravedad actúa hacia abajo)
# ref: https://openseespydoc.readthedocs.io/en/latest/src/eleload.html#eleload-command
ops.eleLoad('-ele', 1, '-type', '-beamUniform', -wu)

### Configuración del motor de análisis (Lineal y Estático)

In [ ]:
# =========================================================================
# CONFIGURACIÓN DEL MOTOR DE ANÁLISIS (Lineal y Estático)
# =========================================================================

# 1. Manejo de restricciones (Boundary Conditions)
# Determina cómo se aplican los apoyos fijos/móviles. 'Transformation' es el método
# más directo y robusto para pasar de la estructura real a las ecuaciones del sistema.
ops.constraints('Transformation')

# 2. Enumerador de grados de libertad
# Organiza la numeración interna de las ecuaciones de rigidez. 'RCM' (Reverse Cuthill-McKee)
# reordena los números para optimizar el ancho de banda de la matriz y acelerar el cálculo.
ops.numberer('RCM')

# 3. Sistema de almacenamiento y resolución lineal
# Define cómo se almacena la matriz de rigidez [K] en memoria y qué solver matemático se usa.
# 'BandGeneral' es ideal para estructuras lineales simples de barras.
ops.system('BandGeneral')

# 4. Algoritmo de solución
# Elige la estrategia para resolver el sistema de ecuaciones. Como nuestra viga es 100% elástica,
# el sistema se resuelve en un único paso lineal directo. Usamos 'Linear'.
ops.algorithm('Linear')

# 5. Integrador del análisis
# Define cómo se aplican los pasos de carga. 'LoadControl' con factor 1.0 significa
# que aplicamos el 100% de la carga total calculada (wu) de un solo golpe.
ops.integrator('LoadControl', 1.0)

# 6. Tipo de Análisis
# Define la naturaleza del problema. En este caso, las cargas no varían en el tiempo (Estático).
ops.analysis('Static')

# Ejecutar el análisis (1 único paso es suficiente para un problema lineal)
ops.analyze(1)
print("Análisis estático lineal finalizado con éxito.")

## Determinación de las reacciones

In [ ]:
# Extraer reacciones en los apoyos (Fuerza en Y es el índice 1)
ops.reactions()
R1_y = ops.nodeReaction(1, 2) # Nodo 1, DOF 2 (Y)
R2_y = ops.nodeReaction(2, 2) # Nodo 2, DOF 2 (Y)

# Validación analítica teórica: R = (q * L) / 2
R_teorica = (wu * Lv) / 2.0

print(f"\n--- VALIDACIÓN DE RESULTADOS ---")
print(f"Reacción Nodo 1 (OpenSees): {R1_y:.2f} kN | Teórica: {R_teorica:.2f} kN")
print(f"Reacción Nodo 2 (OpenSees): {R2_y:.2f} kN | Teórica: {R_teorica:.2f} kN")

## Graficos

In [ ]:
import matplotlib.pyplot as plt
import opsvis as opsv

# 1. Configurar el tamaño y estilo de los gráficos
plt.figure(figsize=(12, 5))

# ==========================================
# ACCIONES SOBRE LA VIGA)
# ==========================================
opsv.plot_model()
plt.title("Esquema estructural")

# ==========================================
# ACCIONES SOBRE LA VIGA)
# ==========================================
opsv.plot_load()
plt.title("Fuerzas distribuidas (w)")


# ==========================================
# DIAGRAMA DE MOMENTOS FLECTORES (M)
# ==========================================
opsv.section_force_diagram_2d('M', sfac=0.05)
plt.title("Diagrama de Momentos Flectores (M)")
plt.grid(True)

# ==========================================
# DIAGRAMA DE ESFUERZOS CORTANTES (V)
# ==========================================
opsv.section_force_diagram_2d('V', sfac=0.05)
plt.title("Diagrama de Esfuerzos Cortantes (V)")
plt.grid(True)

# ==========================================
# DEFORMACIÓN DE LA VIGA (f)
# ==========================================
opsv.plot_defo(1000)
plt.title("Deformada de la viga (Exagerada)")
plt.grid(True)

# Mostrar las figuras de Matplotlib insertadas en el notebook
plt.tight_layout()
plt.show()